# ATLAS-18way setting — wheels for offline train

Same pattern as [`name-that-llm-gemma2b-setting.ipynb`](name-that-llm-gemma2b-setting.ipynb): run **once** on Kaggle with **internet ON** (no GPU). It `pip download`s the packages [`ATLAS-18way-train.ipynb`](ATLAS-18way-train.ipynb) needs (`peft`, `bitsandbytes`, `accelerate`, `transformers`) into `/kaggle/working/wheels`.

Then **Add Input** this notebook’s output on the train kernel (internet off). The train notebook installs with `--no-index --find-links`.

Do **not** download Qwen2.5-7B here. Attach the Kaggle model `qwen-lm/qwen2.5` variation `7b` on the train notebook.

Run this on Kaggle, not locally, so the wheels match the train image (Python / manylinux).

| Package | Pin |
| --- | --- |
| transformers | 4.51.3 |
| bitsandbytes | 0.45.5 |
| accelerate | 1.6.0 |
| peft | 0.15.2 |

Torch / nvidia / triton wheels are deleted after download. Kaggle already has those.


In [ ]:
!pip download "transformers==4.51.3" "bitsandbytes==0.45.5" "accelerate==1.6.0" "peft==0.15.2" \
    -d /kaggle/working/wheels
!rm -f /kaggle/working/wheels/torch-*.whl /kaggle/working/wheels/nvidia_*.whl /kaggle/working/wheels/triton-*.whl
!ls /kaggle/working/wheels


In [ ]:
from pathlib import Path
import json

wheels = Path("/kaggle/working/wheels")
files = sorted(p.name for p in wheels.glob("*.whl"))
need = ("transformers", "bitsandbytes", "accelerate", "peft")
missing = [n for n in need if not any(f.startswith(n) for f in files)]
assert not missing, f"download missed {missing}; files={files[:20]}"

manifest = {
    "purpose": "offline wheels for ATLAS-18way-train.ipynb",
    "packages": {
        "transformers": "4.51.3",
        "bitsandbytes": "0.45.5",
        "accelerate": "1.6.0",
        "peft": "0.15.2",
    },
    "wheels": files,
}
(wheels / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
print(f"{len(files)} wheels -> {wheels}")
print("\n".join(files))
